In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    # Start clean: no login, no stored session
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    time.sleep(1)

    # Real protected route: staff app requires a token (verified: App.jsx token gate).
    # Anonymous users get the Login page instead (App.jsx renders <Login />).
    driver.get("http://localhost:5173/pharmacist/dashboard")
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    time.sleep(1)
    body = driver.find_element(By.TAG_NAME, "body").text
    assert "Welcome back" in body, "Login page not shown to unauthenticated user."
    assert not driver.find_elements(By.XPATH, "//nav[@aria-label='Staff']"), \
        "Protected staff UI exposed without login."
    print("Unauthenticated /pharmacist/dashboard shows: Login (Welcome back).")

    # Same check on the admin route
    driver.get("http://localhost:5173/admin/dashboard")
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    assert not driver.find_elements(By.XPATH, "//nav[@aria-label='Staff']"), \
        "Protected staff UI exposed without login."
    print("Unauthenticated /admin/dashboard shows: Login.")
    print("PASS: Unauthorized page access blocked")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("47_unauthorized_FAIL.png")
finally:
    driver.quit()